# Fasih-SM Fetcher
Untuk menarik data dari Fasih-SM sebagai cadangan data di satuan kerja.

Made with ❤️🩸💦, Gilang Wahyu Prasetyo © BPS Kabupaten Tabalong

# 1. Improt 

## main

In [3]:
import os
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from urllib.parse import unquote
from datetime import datetime, timedelta
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# bikin dulu file ".env" dan isi dengan nilai di bawah ini
# FASIH_USER=username sso
# FASIH_PASS=password sso
load_dotenv()


True

## formatter tanggal biar cakep

In [1]:
def format_fasih_date(date_str, timezone_label="WITA"):
    if not date_str or date_str == "-":
        return "-"
    
    tz_offset = {"WIB": 7, "WITA": 8, "WIT": 9}
    offset = tz_offset.get(timezone_label, 8)
    
    try:
        # Masalahnya di sini: Fasih kirim nanodetik (9 digit), Python cuma support mikro (6 digit).
        # Kita ambil 19 karakter pertama (YYYY-MM-DDTHH:MM:SS)
        clean_date = date_str.split(".")[0] 
        if "T" not in clean_date:
            return date_str # Kembalikan asli jika format tidak sesuai
            
        dt = datetime.strptime(clean_date, "%Y-%m-%dT%H:%M:%S")
        dt_local = dt + timedelta(hours=offset)
        
        return dt_local.strftime(f"%d %b %Y at %H.%M {timezone_label}")
    except Exception as e:
        # Jika gagal parsing, kita coba cara aman: ambil substring tanggal saja
        try:
            return f"{date_str[:10]} (Raw)"
        except:
            return date_str

# 2. Ambil cookie

In [5]:
# Konfigurasi & Inisialisasi
USERNAME = os.getenv("FASIH_USER")
PASSWORD = os.getenv("FASIH_PASS")
OAUTH_LOGIN_URL = "https://fasih-sm.bps.go.id/oauth_login.html"

def build_driver(headless=False):
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    if headless: options.add_argument("--headless=new")
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_cdp_cmd("Network.enable", {})
    return driver

def get_authenticated_session(driver):
    session = requests.Session()
    for cookie in driver.get_cookies():
        session.cookies.set(cookie['name'], cookie['value'])
    bearer_token = driver.execute_script("return window.localStorage.getItem('token');")
    csrf_token = session.cookies.get('XSRF-TOKEN')
    auth_headers = {
        "User-Agent": driver.execute_script("return navigator.userAgent;"),
        "Accept": "application/json, text/plain, */*",
        "X-XSRF-TOKEN": unquote(csrf_token) if csrf_token else "",
        "Authorization": f"Bearer {bearer_token.replace('"', '')}" if bearer_token else ""
    }
    session.headers.update(auth_headers)
    return session

# Eksekusi Login
driver = build_driver(headless=False)
driver.get(OAUTH_LOGIN_URL)

# Alur Klik SSO & Input Login (Diringkas)
wait = WebDriverWait(driver, 30)
sso_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.login-button")))
driver.execute_script("arguments[0].click();", sso_btn)

wait.until(lambda d: "sso.bps.go.id" in d.current_url)
wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys(USERNAME)
pass_el = driver.find_element(By.ID, "password")
pass_el.send_keys(PASSWORD)
pass_el.send_keys(Keys.ENTER)

wait.until(lambda d: "fasih-sm.bps.go.id" in d.current_url)
print("Login Berhasil.")
time.sleep(5) # Tunggu sinkronisasi token ke localStorage

# Ambil Session untuk Scraping Cepat
api_session = get_authenticated_session(driver)

Login Berhasil.


# Kill all chrome running

In [ ]:
driver.quit()
os.system("taskkill /f /im chromedriver.exe /t")
os.system("taskkill /f /im chrome.exe /t")

# 3. Print list of surveys

## def

In [6]:
def fetch_list_surveys(session, survey_type="Pencacahan", page_size=100):
    url = f"https://fasih-sm.bps.go.id/survey/api/v1/surveys/datatable?surveyType={survey_type}"
    payload = {
        "pageNumber": 0,
        "pageSize": page_size,
        "sortBy": "CREATED_AT",
        "sortDirection": "DESC",
        "keywordSearch": ""
    }
    
    response = session.post(url, json=payload)
    if response.status_code == 200:
        data = response.json()
        return data.get('data', {}).get('content', [])
    return []

def process_surveys_to_df(rows, tz="WITA"):
    if not rows: return pd.DataFrame()
    surveys_list = []
    for item in rows:
        surveys_list.append({
            "judul_survei": item.get("name"), # Kolom ke-2
            "id": item.get("id"),
            "unit": item.get("unit"),
            "dibuat_pada": format_fasih_date(item.get("createdAt"), timezone_label=tz)
        })
    
    df = pd.DataFrame(surveys_list)
    # Geser kolom judul_survei ke posisi ke-2 (index 1)
    cols = ['judul_survei', 'id', 'unit', 'dibuat_pada']
    df = df[cols]
    
    # Ubah index agar mulai dari 1
    df.index = df.index + 1
    return df

## implementasi

In [7]:
# 1. Menu Pilihan
print("Pilih Tipe Survei:")
print("1. Pencacahan\n2. Pelatihan\n3. Uji Coba")
pilihan = input("Masukkan angka pilihan (1/2/3): ")

mapping_type = {"1": "Pencacahan", "2": "Pelatihan", "3": "Ujicoba"}
target_type = mapping_type.get(pilihan)

if target_type:
    # 2. Jalankan Fetch
    raw_surveys = fetch_list_surveys(api_session, survey_type=target_type)
    
    # 3. Tampilkan dengan nama baru: datasurveys
    datasurveys = process_surveys_to_df(raw_surveys, tz="WITA")
    
    if not datasurveys.empty:
        print(f"\nSukses! Ditemukan {len(datasurveys)} survei tipe {target_type}.")
        display(datasurveys.head(len(datasurveys)))
    else:
        print(f"Data {target_type} kosong.")
else:
    print("Pilihan tidak valid. Silakan jalankan ulang cell ini.")

Pilih Tipe Survei:
1. Pencacahan
2. Pelatihan
3. Uji Coba


Masukkan angka pilihan (1/2/3):  1



Sukses! Ditemukan 13 survei tipe Pencacahan.


,judul_survei,id,unit,dibuat_pada
1,GC PBI 2026 [TAHAP 2] - PENDATAAN,8712a6fc-a996-4a8f-ad6f-56a278c19288,BADAN PUSAT STATISTIK,03 Mar 2026 at 16.15 WITA
2,SAKERNAS FEB 2026 - PENDATAAN,9b637b4c-2839-4a16-9023-1a62c364572b,BADAN PUSAT STATISTIK,22 Jan 2026 at 16.36 WITA
3,SAKERNAS FEB 2026 - PEMUTAKHIRAN,aa4e5f1a-6329-4861-8f83-6c5d2769056b,BADAN PUSAT STATISTIK,21 Jan 2026 at 23.03 WITA
4,SKP 2025,4adacbc5-c8d0-47fb-b0fc-227ffa10c2e7,BADAN PUSAT STATISTIK,14 Sep 2025 at 09.20 WITA
5,SURVEI POLA DISTRIBUSI BARANG TAHUN 2025,30ac6046-2e71-4f38-ab37-af627201941a,BADAN PUSAT STATISTIK,30 Jun 2025 at 10.43 WITA
6,SURVEI PERDAGANGAN BARANG DOMESTIK 2025 - PAW,dbecd6c2-c7cc-447c-a4dc-a7dd8d34cf05,BADAN PUSAT STATISTIK,02 Jun 2025 at 02.26 WITA
7,VPEK 2025,9fc084cf-921d-4011-b6bd-3c38a07130b1,BADAN PUSAT STATISTIK,27 May 2025 at 17.48 WITA
8,PENCACAHAN SPUNP - O 2025,8428dd2f-15f1-4988-9154-4b8fd263ff17,BADAN PUSAT STATISTIK,14 Apr 2025 at 15.09 WITA
9,PENCACAHAN SPUNP - J 2025,e2d07fc4-9ffe-487f-9606-08d68a79e210,BADAN PUSAT STATISTIK,14 Apr 2025 at 15.06 WITA
10,PENCACAHAN SPUNP - I 2025,ade5f14c-29ed-4ae3-9802-541e80610b51,BADAN PUSAT STATISTIK,14 Apr 2025 at 15.03 WITA


# 4. ambil metadata (data survey settings)

## def

In [8]:
def fetch_json(session, url):
    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()
        res = r.json()
        return res.get("data") if res.get("data") is not None else {}
    except:
        return {}

def fetch_full_survey_settings_flat(session, survey_id):
    det = fetch_json(session, f"https://fasih-sm.bps.go.id/survey/api/v1/surveys/{survey_id}")
    per = fetch_json(session, f"https://fasih-sm.bps.go.id/survey/api/v1/survey-periods?surveyId={survey_id}")
    
    per_list = per if isinstance(per, list) else []
    region_id = det.get('regionGroupId')
    reg = fetch_json(session, f"https://fasih-sm.bps.go.id/region/api/v1/region-metadata?id={region_id}") if region_id else {}

    act_per = next((p for p in per_list if p.get("isActive")), per_list[0] if per_list else {})

    meta = {
        "judul": det.get("name", "-"),
        "tipe": det.get("surveyType", "-"),
        "mode": ", ".join([m.get("mode", "") for m in det.get("surveyModeList", [])]) if det.get("surveyModeList") else "-",
        "wilayah_ver": reg.get("groupName", "-"),
        "level_wilayah": " > ".join([l.get("name", "") for l in reg.get("level", [])]) if reg.get("level") else "-",
        "jenis_panel": "Panel" if det.get("panelType") else "Non-Panel",
        "jenis_pencacah": "Banyak" if det.get("isMultiPencacah") else "Satu",
        "periode_aktif": act_per.get("name", "-"),
        "tgl_mulai": act_per.get("startDate"),
        "tgl_selesai": act_per.get("endDate"),
        "id_periode": act_per.get("id", "-")
    }
    # Opsional: Tambahkan semua field asli jika ingin sangat lengkap
    # meta.update({f"raw_{k}": v for k, v in det.items() if not isinstance(v, (list, dict))})
    
    return [meta]

## implementasi


In [12]:
if 'datasurveys' in locals() and not datasurveys.empty:
    print(f"\nMenampilkan {len(datasurveys)} daftar survei.")
    display(datasurveys)
    
    pilihan_idx = input("\nMasukkan Nomor Indeks survei (1, 2, dst): ")
    
    try:
        idx = int(pilihan_idx)
        # Ambil data berdasarkan index yang sudah kita set (1-based)
        survey_id_target = datasurveys.loc[idx, 'id']
        judul_terpilih = datasurveys.loc[idx, 'judul_survei']
        
        print(f"\n--- Memuat Metadata untuk: {judul_terpilih} ---")
        settings = fetch_full_survey_settings_flat(api_session, survey_id_target)
        
        if settings:
            item = settings[0]
            item['tgl_mulai'] = format_fasih_date(item.get('tgl_mulai'), timezone_label="WITA")
            item['tgl_selesai'] = format_fasih_date(item.get('tgl_selesai'), timezone_label="WITA")
            current_period_id = item.get("id_periode")
            
            print("-" * 50)
            for k, v in item.items():
                print(f"{k.ljust(20)} : {v}")
            print("-" * 50)
            print(f"ID Periode Aktif terkunci: {current_period_id}")
        else:
            print("Gagal menarik metadata.")
            
    except ValueError:
        print("Input harus berupa angka.")
    except KeyError:
        print(f"Indeks {pilihan_idx} tidak ada dalam tabel.")
else:
    print("Jalankan Bagian 3 dulu bos.")


Menampilkan 13 daftar survei.


,judul_survei,id,unit,dibuat_pada
1,GC PBI 2026 [TAHAP 2] - PENDATAAN,8712a6fc-a996-4a8f-ad6f-56a278c19288,BADAN PUSAT STATISTIK,03 Mar 2026 at 16.15 WITA
2,SAKERNAS FEB 2026 - PENDATAAN,9b637b4c-2839-4a16-9023-1a62c364572b,BADAN PUSAT STATISTIK,22 Jan 2026 at 16.36 WITA
3,SAKERNAS FEB 2026 - PEMUTAKHIRAN,aa4e5f1a-6329-4861-8f83-6c5d2769056b,BADAN PUSAT STATISTIK,21 Jan 2026 at 23.03 WITA
4,SKP 2025,4adacbc5-c8d0-47fb-b0fc-227ffa10c2e7,BADAN PUSAT STATISTIK,14 Sep 2025 at 09.20 WITA
5,SURVEI POLA DISTRIBUSI BARANG TAHUN 2025,30ac6046-2e71-4f38-ab37-af627201941a,BADAN PUSAT STATISTIK,30 Jun 2025 at 10.43 WITA
6,SURVEI PERDAGANGAN BARANG DOMESTIK 2025 - PAW,dbecd6c2-c7cc-447c-a4dc-a7dd8d34cf05,BADAN PUSAT STATISTIK,02 Jun 2025 at 02.26 WITA
7,VPEK 2025,9fc084cf-921d-4011-b6bd-3c38a07130b1,BADAN PUSAT STATISTIK,27 May 2025 at 17.48 WITA
8,PENCACAHAN SPUNP - O 2025,8428dd2f-15f1-4988-9154-4b8fd263ff17,BADAN PUSAT STATISTIK,14 Apr 2025 at 15.09 WITA
9,PENCACAHAN SPUNP - J 2025,e2d07fc4-9ffe-487f-9606-08d68a79e210,BADAN PUSAT STATISTIK,14 Apr 2025 at 15.06 WITA
10,PENCACAHAN SPUNP - I 2025,ade5f14c-29ed-4ae3-9802-541e80610b51,BADAN PUSAT STATISTIK,14 Apr 2025 at 15.03 WITA



Masukkan Nomor Indeks survei (1, 2, dst):  2



--- Memuat Metadata untuk: SAKERNAS FEB 2026 - PENDATAAN ---
--------------------------------------------------
judul                : SAKERNAS FEB 2026 - PENDATAAN
tipe                 : Pencacahan
mode                 : CAPI
wilayah_ver          : MFD SAKERNAS FEB 2026
level_wilayah        : PROVINSI > KABUPATEN/KOTA > KECAMATAN > DESA > SLS > SUBSLS
jenis_panel          : Non-Panel
jenis_pencacah       : Satu
periode_aktif        : FEBRUARI
tgl_mulai            : 2026-02-07
tgl_selesai          : 2026-03-11
id_periode           : 1e42622b-d016-40c1-b824-fdcc6868fc3a
--------------------------------------------------
ID Periode Aktif terkunci: 1e42622b-d016-40c1-b824-fdcc6868fc3a


# 5. ambil data petugas (terkunci berdasar metadata)
kalo mo ganti survei, ganti di bagian 4

## def

In [10]:
def fetch_petugas_automated(session, survey_id, period_id):
    """
    Otomatis mencari Role ID Pencacah dan menarik data petugasnya.
    """
    # 1. Discovery Role ID & Group ID berdasar survey_id_target
    role_url = f"https://fasih-sm.bps.go.id/survey/api/v1/survey-roles?surveyId={survey_id}"
    role_res = session.get(role_url)
    
    target_role_id = None
    target_group_id = None
    
    if role_res.status_code == 200:
        roles_data = role_res.json().get("data", [])
        # Cari role 'Pencacah' atau yang memiliki flag isPencacah=True
        for r in roles_data:
            if r.get("description") == "Pencacah" or r.get("isPencacah") is True:
                target_role_id = r.get("id")
                target_group_id = r.get("surveyRoleGroupId")
                break
    
    if not target_role_id:
        return {"error": f"Tidak ditemukan Role Pencacah untuk survey {survey_id}"}

    # 2. Tarik Data Petugas menggunakan ID yang ditemukan dan current_period_id
    api_petugas = f"https://fasih-sm.bps.go.id/analytic/api/v2/survey-period-role-user/datatable"
    params = {
        "surveyPeriodId": period_id,
        "surveyRoleGroupId": target_group_id,
        "surveyRoleId": target_role_id
    }
    
    # URL dengan query params
    api_url_full = f"{api_petugas}?surveyPeriodId={params['surveyPeriodId']}&surveyRoleGroupId={params['surveyRoleGroupId']}&surveyRoleId={params['surveyRoleId']}"
    
    payload = {
        "pageNumber": 0, 
        "pageSize": 100, 
        "sortBy": "ID", 
        "sortDirection": "ASC", 
        "keywordSearch": ""
    }
    
    response = session.post(api_url_full, json=payload)
    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"Gagal fetch petugas: {response.status_code}"}

def process_petugas_to_df(json_data):
    if not json_data or "data" not in json_data or "searchData" not in json_data["data"]:
        return pd.DataFrame()

    rows = []
    search_data = json_data["data"]["searchData"]
    for item in search_data:
        user = item.get("user", {})
        # Ambil smallestRegionCode dari list objects
        regions = [r.get("smallestRegionCode") for r in item.get("smallestRegionCodes", [])]
        
        rows.append({
            "Fullname": user.get("fullname") or "-",
            "Email": user.get("email"),
            "Role": item.get("surveyRole", {}).get("description"),
            "Wilayah": ", ".join([str(r) for r in regions]) if regions else "-"
        })
    
    df = pd.DataFrame(rows)
    if not df.empty:
        df.index = range(1, len(df) + 1)
    return df

## implementasi

In [13]:
# Pastikan Bagian 4 sudah dijalankan dan variabel ini tersedia
if 'survey_id_target' in locals() and 'current_period_id' in locals():
    print("=" * 60)
    print(f"MENARIK DATA PETUGAS: {judul_terpilih}")
    print(f"Survey ID: {survey_id_target}")
    print(f"Period ID: {current_period_id}")
    print("=" * 60)
    
    # 1. Jalankan Discovery & Fetch secara otomatis
    raw_petugas = fetch_petugas_automated(api_session, survey_id_target, current_period_id)
    
    if isinstance(raw_petugas, dict) and "error" in raw_petugas:
        print(f"Gagal: {raw_petugas['error']}")
    else:
        # 2. Tampilkan Hasil
        datapetugas = process_petugas_to_df(raw_petugas)
        if not datapetugas.empty:
            print(f"Berhasil! Ditemukan {len(datapetugas)} petugas.")
            display(datapetugas)
        else:
            print("Data petugas kosong atau tidak ditemukan.")
else:
    print("Error: Silakan jalankan Langkah 4 (Pilih Survei) terlebih dahulu agar ID tersedia.")

MENARIK DATA PETUGAS: SAKERNAS FEB 2026 - PENDATAAN
Survey ID: 9b637b4c-2839-4a16-9023-1a62c364572b
Period ID: 1e42622b-d016-40c1-b824-fdcc6868fc3a
Berhasil! Ditemukan 8 petugas.


,Fullname,Email,Role,Wilayah
1,-,itscaca2@gmail.com,PPL,"6309050011000500, 6309030016000401"
2,NurwidaSiti,nurwindasitisarah20@gmail.com,PPL,"6309060014000700, 6309070006002100"
3,Fathul,fathulmarhamah04@gmail.com,PPL,"6309080036000300, 6309080004000100"
4,-,lidarusmi10@gmail.com,PPL,"6309030008000201, 6309020007000100"
5,PUTRILAURENCI,amianniputri@gmail.com,PPL,"6309070001000300, 6309070005001702"
6,Nanik,nanikyuyun30@gmail.com,PPL,"6309100011000300, 6309100001000800"
7,Yati,yatikamariah15@gmail.com,PPL,"6309010004000300, 6309010027000600"
8,Usratul,usratulusroh@gmail.com,PPL,"6309070003001802, 6309070003000101"


# 6. Ambil ringkasan sampel (terkunci berdasar metadata)
kalo mo ganti survei, ganti di bagian 4

## def

In [14]:
def process_analytic_to_df(json_data, tz="WITA"):
    if not json_data or "searchData" not in json_data:
        return pd.DataFrame()
    
    rows = []
    for item in json_data["searchData"]:
        # Bongkar hirarki wilayah
        reg = item.get("region", {})
        lvl3 = reg.get("level1", {}).get("level2", {}).get("level3", {})
        lvl4 = lvl3.get("level4", {}) if lvl3 else {}
        lvl5 = lvl4.get("level5", {}) if lvl4 else {}
        lvl6 = lvl5.get("level6", {}) if lvl5 else {}
        
        rows.append({
            "Sample ID": item.get("id"),
            "ID SLS": item.get("codeIdentity"),
            "Kepala Keluarga": item.get("data1"),
            "Anggota Keluarga": item.get("data2"),
            "Alamat": item.get("data3"),
            "Status Keberadaan": item.get("data4"),
            "Status Dokumen": item.get("assignmentStatusAlias"),
            "Created": format_fasih_date(item.get("dateCreated"), tz),
            "Modified": format_fasih_date(item.get("dateModified"), tz),
            "Latitude": item.get("latitude"),
            "Longitude": item.get("longitude"),
            "Pencacah": item.get("currentUserFullname"),
            "Email Pencacah": item.get("currentUserUsername"),
            "Kec": f"{lvl3.get('code', '-')}. {lvl3.get('name', '-')}" if lvl3 else "-",
            "Des": f"{lvl4.get('code', '-')}. {lvl4.get('name', '-')}" if lvl4 else "-",
            "SLS": lvl5.get('name', '-') if lvl5 else "-",
            "Sub SLS": lvl6.get('code', '-') if lvl6 else "-"
        })
    return pd.DataFrame(rows)

## Implementasi

In [15]:
# --- KONFIGURASI ---
n_target = 50      # Total data yang ingin diambil (Sesuai totalHit di log)
batch_size = 25      # Pilihan: 10, 25, 50, 100
timezone_set = "WITA"
# -------------------

if 'current_period_id' in locals() and current_period_id:
    url_analytic = "https://fasih-sm.bps.go.id/analytic/api/v2/assignment/datatable-all-user-survey-periode"
    all_results = []
    start_idx = 0
    draw_count = 1
    
    print("=" * 60)
    print(f"MENARIK RINGKASAN SAMPEL: {judul_terpilih}")
    print("=" * 60)

    while start_idx < n_target:
        # Payload lengkap sesuai log browser
        payload = {
            "draw": draw_count,
            "columns": [
                {"data": "id", "name": "", "searchable": True, "orderable": False, "search": {"value": "", "regex": False}},
                {"data": "codeIdentity", "name": "", "searchable": True, "orderable": False, "search": {"value": "", "regex": False}},
                {"data": "data1", "name": "", "searchable": True, "orderable": True, "search": {"value": "", "regex": False}},
                {"data": "data2", "name": "", "searchable": True, "orderable": True, "search": {"value": "", "regex": False}},
                {"data": "data3", "name": "", "searchable": True, "orderable": True, "search": {"value": "", "regex": False}},
                {"data": "data4", "name": "", "searchable": True, "orderable": True, "search": {"value": "", "regex": False}}
            ],
            "order": [{"column": 0, "dir": "asc"}],
            "start": start_idx,
            "length": batch_size,
            "search": {"value": "", "regex": False},
            "assignmentExtraParam": {
                "surveyPeriodId": current_period_id,
                "assignmentErrorStatusType": -1,
                "filterTargetType": "TARGET_ONLY"
            }
        }
        
        try:
            resp = api_session.post(url_analytic, json=payload, timeout=30)
            
            if resp.status_code == 200:
                raw_data = resp.json()
                # Update n_target secara dinamis jika totalHit di server berbeda
                total_server = raw_data.get("totalHit", n_target)
                n_target = min(n_target, total_server)
                
                df_batch = process_analytic_to_df(raw_data, tz=timezone_set)
                all_results.append(df_batch)
                
                print(f"[{min(start_idx + batch_size, n_target)}/{n_target}] Berhasil menarik {len(df_batch)} data...")
                
                start_idx += batch_size
                draw_count += 1
                time.sleep(0.4) # Jeda aman agar tidak dianggap spam
            else:
                print(f"Gagal! Status: {resp.status_code} - {resp.text}")
                break
        except Exception as e:
            print(f"Error: {str(e)}")
            break

    if all_results:
        datapreviewsample = pd.concat(all_results, ignore_index=True)
        datapreviewsample.index = range(1, len(datapreviewsample) + 1)
        print(f"\n--- SELESAI: {len(datapreviewsample)} DATA TERKUMPUL ---")
        display(datapreviewsample.head(10))
    else:
        print("Data gagal ditarik.")
else:
    print("Pilih survei dulu di Bagian 4.")

MENARIK RINGKASAN SAMPEL: SAKERNAS FEB 2026 - PENDATAAN
[25/50] Berhasil menarik 25 data...
[50/50] Berhasil menarik 25 data...

--- SELESAI: 50 DATA TERKUMPUL ---


,Sample ID,ID SLS,Kepala Keluarga,Anggota Keluarga,Alamat,Status Keberadaan,Status Dokumen,Created,Modified,Latitude,Longitude,Pencacah,Email Pencacah,Kec,Des,SLS,Sub SLS
1,fce8f4ac-1773-4341-aee9-bd8fc4509310,6309080036000300 - 7,7,ASMAH,103,MARINDI RT 003,COMPLETED BY Admin Provinsi,05 Feb 2026 at 08.06 WITA,11 Mar 2026 at 10.10 WITA,-2.012495,115.610930,Sari Ayutyas,sariayutyas,080. HARUAI,036. MARINDI,RT 003,00
2,a3a363fa-3201-4b53-954c-ae8f533f408a,6309030008000201 - 2,2,UMAR,49,JL. A. YANI,COMPLETED BY Admin Provinsi,04 Feb 2026 at 14.42 WITA,11 Mar 2026 at 10.06 WITA,-2.268798,115.291420,Galih Saputro,galih.saputro,030. KELUA,008. TAKULAT,RT 002,01
3,64ce064d-373a-48d4-9b46-1f25f138d33d,6309020007000100 - 1,1,MURDIAN,9,JL A.YANI,COMPLETED BY Admin Provinsi,04 Feb 2026 at 14.42 WITA,11 Mar 2026 at 10.05 WITA,-2.308040,115.295555,Galih Saputro,galih.saputro,020. PUGAAN,007. SUNGAI RUKAM I,RT 001,00
4,d077fef9-96c3-4b98-93f6-9189d7c5953c,6309010004000300 - 7,7,ALAMSYAH,28,SEI DURIAN,COMPLETED BY Admin Provinsi,04 Feb 2026 at 14.42 WITA,11 Mar 2026 at 10.02 WITA,-2.351897,115.280205,Galih Saputro,galih.saputro,010. BANUA LAWAS,004. SUNGAI DURIAN,RT 003,00
5,cbd50c5d-92c4-44fc-b162-707fdd057c98,6309070006002100 - 8,8,NOOR DIANA,59,UNGGUNG,COMPLETED BY Admin Provinsi,08 Feb 2026 at 17.16 WITA,09 Mar 2026 at 12.09 WITA,-2.160678,115.434425,Annisa Auliarahmah,annisa.auliarahmah,070. MURUNG PUDAK,006. BELIMBING,RT 021,00
6,b5001d48-5d18-4642-b9a5-8e717db36458,6309080004000100 - 7,7,SAHRUJI,94,SERADANG,COMPLETED BY Admin Provinsi,05 Feb 2026 at 08.06 WITA,09 Mar 2026 at 11.55 WITA,-2.047842,115.560370,Sari Ayutyas,sariayutyas,080. HARUAI,004. SERADANG,RT 001,00
7,4e023363-62e5-499a-8cfc-586482879a9b,6309010027000600 - 9,9,HARIS FADLY,9,SEI ANYAR,COMPLETED BY Admin Provinsi,04 Feb 2026 at 14.42 WITA,09 Mar 2026 at 11.51 WITA,-2.312730,115.286285,Galih Saputro,galih.saputro,010. BANUA LAWAS,027. SUNGAI ANYAR,RT 006,00
8,036c8fdf-f9c8-4260-9469-843b46cb8827,6309070006002100 - 10,10,YULI WAHYONO,53,JL. GUNUNG BATU,COMPLETED BY Admin Provinsi,08 Feb 2026 at 17.16 WITA,09 Mar 2026 at 09.26 WITA,-2.160660,115.434420,Annisa Auliarahmah,annisa.auliarahmah,070. MURUNG PUDAK,006. BELIMBING,RT 021,00
9,29b09e86-2d69-4adf-8eb9-515dc618f92c,6309070006002100 - 9,9,EDNA GALIS NANDHO,96,PANAAN,COMPLETED BY Admin Provinsi,08 Feb 2026 at 17.16 WITA,09 Mar 2026 at 09.03 WITA,-2.159719,115.432304,Annisa Auliarahmah,annisa.auliarahmah,070. MURUNG PUDAK,006. BELIMBING,RT 021,00
10,325c2f23-4a7a-40d0-8a2e-11e3ea941eae,6309070006002100 - 1,1,YUSNI,1,BELIMBING,COMPLETED BY Admin Provinsi,08 Feb 2026 at 17.16 WITA,09 Mar 2026 at 01.49 WITA,-2.166109,115.430680,Annisa Auliarahmah,annisa.auliarahmah,070. MURUNG PUDAK,006. BELIMBING,RT 021,00


# 7. ambil rincian sampel (terkunci berdasar metadata)
kalo mo ganti survei, ganti di bagian 4

### def

In [16]:
def parse_detail_sample(json_response):
    if not json_response or not json_response.get("success"):
        return None
    
    raw_data = json_response.get("data", {})
    
    # 1. Base Info (Data tingkat atas)
    result = {
        "Sample ID": raw_data.get("_id"),
        "ID SLS": raw_data.get("code_identity"),
        "Status Dokumen": raw_data.get("assignment_status_alias"),
        "Latitude": raw_data.get("latitude"),
        "Longitude": raw_data.get("longitude"),
        "Petugas Terakhir": raw_data.get("current_user_fullname")
    }
    
    # 2. Parsing pre_defined_data (Data Prelist)
    pre_defined_str = raw_data.get("pre_defined_data", "{}")
    try:
        pre_data = json.loads(pre_defined_str)
        for item in pre_data.get("predata", []):
            key = f"Prelist_{item.get('dataKey')}"
            val = item.get("answer")
            # Jika nested (list/dict), gabungkan jadi string
            result[key] = str(val) if not isinstance(val, (list, dict)) else json.dumps(val)
    except:
        pass

    # 3. Parsing data (Jawaban Survei)
    data_content_str = raw_data.get("data", "{}")
    try:
        content_data = json.loads(data_content_str)
        # Ambil info meta dari level data
        result["Waktu Submit"] = content_data.get("updatedAt")
        
        for ans in content_data.get("answers", []):
            key = f"Ans_{ans.get('dataKey')}"
            val = ans.get("answer")
            
            # Gabungkan jika jawaban berupa list of objects (seperti status_keberadaan)
            if isinstance(val, list):
                # Ekstrak label jika ada, jika tidak dump ke string
                text_ans = ", ".join([str(v.get('label', v)) if isinstance(v, dict) else str(v) for v in val])
                result[key] = text_ans
            else:
                result[key] = val
    except:
        pass
        
    return result

### implementasi

In [ ]:
pd.set_option('display.max_columns', None)
datadetailsample

In [19]:
# --- KONFIGURASI RESILIENCE ---
# all_detailed_rows = [] # KOMEN INI KALO RTO 
start_from_index = 0   # GANTI INI jika RTO. Misal mati di 500, isi dengan 500.
n_to_fetch = 100       # Jumlah data yang mau diambil dari titik start
# ------------------------------

if 'datapreviewsample' in locals() and not datapreviewsample.empty:
    # Ambil list ID dan potong berdasarkan titik mulai
    full_list_ids = datapreviewsample["Sample ID"].tolist()
    list_sample_ids = full_list_ids[start_from_index : start_from_index + n_to_fetch]
    
    total_ids = len(list_sample_ids)
    
    # all_detailed_rows jangan direset ke [] kalau mau digabung dalam satu sesi running
    # Tapi lebih aman biarkan per batch agar memory tidak bengkak
    current_batch_rows = []

    if start_from_index == 0:
        print(f"--- START FETCH ---")
    else :
        print(f"--- RESUME FETCH FROM index {start_from_index+1} ---")
    # print(f"Memulai dari index: {start_from_index}")
    print(f"Target batch ini: {total_ids} sampel")
    print("=" * 30)

    for i, s_id in enumerate(list_sample_ids, 1):
        api_url = f"https://fasih-sm.bps.go.id/assignment-general/api/assignment/get-by-id-with-data-for-scm?id={s_id}"
        
        try:
            # Tambahkan timeout eksplisit agar script tidak gantung
            response = api_session.get(api_url, timeout=15)
            
            if response.status_code == 200:
                json_detail = response.json()
                row_data = parse_detail_sample(json_detail)
                
                if row_data:
                    current_batch_rows.append(row_data)
                    # Menampilkan index asli dari dataframe awal agar mudah tracking
                    print(f"[{start_from_index + i}] Berhasil: {s_id}")
                else:
                    print(f"[{start_from_index + i}] Gagal parsing: {s_id}")
            else:
                print(f"[{start_from_index + i}] Gagal API ({response.status_code}): {s_id}")
                
        except Exception as e:
            print(f"[{start_from_index + i}] ERROR (RTO/Koneksi): {str(e)}")
            print("Berhenti di sini. Silakan catat index terakhir dan jalankan ulang.")
            break # Stop loop jika terjadi error koneksi berat
    
        time.sleep(0.4)

    # Gabungkan hasil batch ini ke storage utama
    if current_batch_rows:
        if 'all_detailed_rows' not in locals():
            all_detailed_rows = []
            
        all_detailed_rows.extend(current_batch_rows)
        datadetailsample = pd.DataFrame(all_detailed_rows)
        
        jml_kolom = len(datadetailsample.columns)
        print(f"\n--- PENARIKAN SELESAI ---")
        print(f"Total data: {len(datadetailsample)}")
        # Hitung jumlah kolom
        
        print(f"Total Kolom Terbentuk: {jml_kolom}")
        print("-" * 30)
        display(datadetailsample.tail(10))
else:
    print("DataFrame ringkasan kosong.")

--- START FETCH ---
Target batch ini: 50 sampel
[1] Berhasil: fce8f4ac-1773-4341-aee9-bd8fc4509310
[2] Berhasil: a3a363fa-3201-4b53-954c-ae8f533f408a
[3] Berhasil: 64ce064d-373a-48d4-9b46-1f25f138d33d
[4] Berhasil: d077fef9-96c3-4b98-93f6-9189d7c5953c
[5] Berhasil: cbd50c5d-92c4-44fc-b162-707fdd057c98
[6] Berhasil: b5001d48-5d18-4642-b9a5-8e717db36458
[7] Berhasil: 4e023363-62e5-499a-8cfc-586482879a9b
[8] Berhasil: 036c8fdf-f9c8-4260-9469-843b46cb8827
[9] Berhasil: 29b09e86-2d69-4adf-8eb9-515dc618f92c
[10] Berhasil: 325c2f23-4a7a-40d0-8a2e-11e3ea941eae
[11] Berhasil: 1917bab6-0dca-4456-8f73-52af05a370fc
[12] Berhasil: 87a672fe-4f25-4199-9135-5dbb7bc77790
[13] Berhasil: 7f772cbe-1171-4e2e-a8b4-8e8fa03efa68
[14] Berhasil: 75078d4c-9dc5-4f96-9dad-722a65694cd9
[15] Berhasil: 06ee86b3-4569-46f0-bcec-1221d29ab892
[16] Berhasil: e0806d5b-d647-4041-aca2-8aa6fbbb3096
[17] Berhasil: 6b4758e0-a8ed-4c3d-8e38-2351c98c2221
[18] Berhasil: e687ea9a-36c3-43b3-b490-e103f7dfe27a
[19] Berhasil: dbe8321d-1

,Sample ID,ID SLS,Status Dokumen,Latitude,Longitude,Petugas Terakhir,Prelist_id_sakp,Prelist__source_survey_period_id,Prelist_id_prelist,Prelist_prov,...,Ans_mcd_cm_b#1,Ans_agf_mkt#3,Ans_agf_hir#3,Ans_dem_jmlp#4,Ans_dem_nmpl1#4,Ans_dem_kdpl1#4,Ans_dem_pylg1#4,Ans_dem_pl4mg1#4,Ans_srh_alsn_l#1,Ans_dem_mgsert#3
40,f8442929-f0fe-411e-bb0f-bda69a1541e3,6309020007000100 - 10,COMPLETED BY Admin Provinsi,-2.308008,115.295450,Galih Saputro,e7aa7f40-9368-493d-862d-335f78560fab,de57c799-05ef-4d5a-b0a8-d70adfa68090,0023,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,e093a1ca-eaea-4fc6-8c33-8e40a2c32bcc,6309030008000201 - 6,COMPLETED BY Admin Provinsi,-2.269022,115.290970,Galih Saputro,6270f351-b10c-4418-9864-253cf7b59f31,de57c799-05ef-4d5a-b0a8-d70adfa68090,0026,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,565686ad-9841-48fb-92cb-a9b4b5b23737,6309030008000201 - 1,COMPLETED BY Admin Provinsi,-2.269131,115.290660,Galih Saputro,9a1e485b-2491-4895-8213-9976e0039eee,de57c799-05ef-4d5a-b0a8-d70adfa68090,0008,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,0a095aad-a1de-49f8-b120-abb4a61d4763,6309020007000100 - 6,COMPLETED BY Admin Provinsi,-2.307373,115.295590,Galih Saputro,99315b0a-4896-4485-83ad-e08211577584,de57c799-05ef-4d5a-b0a8-d70adfa68090,0016,[63] KALIMANTAN SELATAN,...,"1. YA, JAM MINIMUM UNTUK BEKERJA",4. Seluruhnya untuk dikonsumsi rumah tangga,2. TIDAK,1.0,PELATIHAN INSTALASI LISTRIK,[211] TEKNISI INSTALASI LISTRIK,1. Lembaga Pelatihan Kerja Pemerintah,2. TIDAK,NaN,NaN
44,58ecb8b3-ac35-4f18-a89d-fad9fee86336,6309020007000100 - 5,COMPLETED BY Admin Provinsi,-2.307908,115.294914,Galih Saputro,f24751f5-9015-432b-9dc6-3e6e4d890cb0,de57c799-05ef-4d5a-b0a8-d70adfa68090,0010,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,fdcff539-bb4c-4a0a-a055-ca0dc7598cc4,6309020007000100 - 2,COMPLETED BY Admin Provinsi,-2.307556,115.295500,Galih Saputro,05adc42a-6805-4e6d-ace0-7189e8b9d8fe,de57c799-05ef-4d5a-b0a8-d70adfa68090,0017,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,5ec1d8f2-189b-4a8a-a7f7-821342a1cb21,6309080004000100 - 2,COMPLETED BY Admin Provinsi,-2.051791,115.556960,Sari Ayutyas,7f7417db-4933-40a8-9d1a-418dfd8f09e2,de57c799-05ef-4d5a-b0a8-d70adfa68090,0040,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUDAH TUA,NaN
47,ada5dd83-febd-40cc-a620-33882d4a7161,6309080004000100 - 5,COMPLETED BY Admin Provinsi,-2.046878,115.561264,Sari Ayutyas,e9f3d99f-46a8-4b67-911b-bb7a9a681f4b,de57c799-05ef-4d5a-b0a8-d70adfa68090,0012,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1. YA
48,53c802a0-ab87-48ff-b9d3-75736064a2a7,6309080004000100 - 6,COMPLETED BY Admin Provinsi,-2.051078,115.557420,Sari Ayutyas,3e7e145a-cbfe-4717-98bc-b0777deac8ae,de57c799-05ef-4d5a-b0a8-d70adfa68090,0034,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,b271004a-55b5-45c0-b49f-7cb14cb8dbd8,6309070006002100 - 5,COMPLETED BY Admin Provinsi,-2.161784,115.432274,Annisa Auliarahmah,17be4826-1719-4c5c-bb3d-76a1f308d4bc,de57c799-05ef-4d5a-b0a8-d70adfa68090,0034,[63] KALIMANTAN SELATAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
